# House Prices 


In [1]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error


In [2]:
# 1. Load data
# =====================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

y = train["SalePrice"]
X = train.drop(columns=["SalePrice"])
X_test = test.copy()

In [3]:
# 2. Feature engineering
# =====================
def add_features(df):
    df = df.copy()

    df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"] + df["2ndFlrSF"]

    df["TotalBathrooms"] = (
        df["FullBath"]
        + 0.5 * df["HalfBath"]
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )

    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["IsRemodeled"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)

    df["HasGarage"] = (df["GarageArea"].fillna(0) > 0).astype(int)
    df["HasBasement"] = (df["TotalBsmtSF"].fillna(0) > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)

    df["GrLivArea_log"] = np.log1p(df["GrLivArea"])
    df["LotArea_log"] = np.log1p(df["LotArea"])

    return df
X["Has2ndFloor"] = (X["2ndFlrSF"] > 0).astype(int)
X_test["Has2ndFloor"] = (X_test["2ndFlrSF"] > 0).astype(int)

X["HasPool"] = (X["PoolArea"].fillna(0) > 0).astype(int)
X_test["HasPool"] = (X_test["PoolArea"].fillna(0) > 0).astype(int)

X["HasPorch"] = (
    X["OpenPorchSF"].fillna(0)
    + X["EnclosedPorch"].fillna(0)
    + X["3SsnPorch"].fillna(0)
    + X["ScreenPorch"].fillna(0)
) > 0
X["HasPorch"] = X["HasPorch"].astype(int)

X_test["HasPorch"] = (
    X_test["OpenPorchSF"].fillna(0)
    + X_test["EnclosedPorch"].fillna(0)
    + X_test["3SsnPorch"].fillna(0)
    + X_test["ScreenPorch"].fillna(0)
) > 0
X_test["HasPorch"] = X_test["HasPorch"].astype(int)

X["HasDeck"] = (X["WoodDeckSF"].fillna(0) > 0).astype(int)
X_test["HasDeck"] = (X_test["WoodDeckSF"].fillna(0) > 0).astype(int)

X = add_features(X)
X_test = add_features(X_test)

In [4]:
# 3. Prepare categorical features for CatBoost
# =====================
cat_features = X.select_dtypes(include=["object"]).columns.tolist()
for c in cat_features:
    X[c] = X[c].fillna("Missing").astype(str)
    X_test[c] = X_test[c].fillna("Missing").astype(str)


In [5]:
print(cat_features[:])

['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


In [6]:
# We are saving the numerical version of OverallQual
X["OverallQual_num"] = X["OverallQual"]
X_test["OverallQual_num"] = X_test["OverallQual"]

In [7]:
X["QualityArea"] = X["OverallQual_num"] * X["GrLivArea"]
X_test["QualityArea"] = X_test["OverallQual_num"] * X_test["GrLivArea"]

In [8]:
pseudo_cat = ["MSSubClass", "MoSold", "YrSold", "OverallQual", "OverallCond"]

for c in pseudo_cat:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

In [9]:
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

In [10]:
# 4. Log target
# =====================
y_log = np.log1p(y)

In [11]:
# 5. Cross-validation
# =====================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=6000,
        learning_rate=0.02,
        depth=8,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=200,
        early_stopping_rounds=300
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        use_best_model=True
    )

    val_pred = model.predict(X_val)
    rmse = mean_squared_error(y_val, val_pred, squared=False)
    rmse_scores.append(rmse)

    print(f"Fold {fold+1} RMSE (log): {rmse}")

print("CV RMSE mean:", np.mean(rmse_scores))
print("CV RMSE std:", np.std(rmse_scores))


0:	learn: 0.3853105	test: 0.4280000	best: 0.4280000 (0)	total: 88ms	remaining: 8m 48s
200:	learn: 0.1123425	test: 0.1561510	best: 0.1561510 (200)	total: 2.57s	remaining: 1m 14s
400:	learn: 0.0863753	test: 0.1425185	best: 0.1425185 (400)	total: 5.19s	remaining: 1m 12s
600:	learn: 0.0749952	test: 0.1397769	best: 0.1397769 (600)	total: 7.99s	remaining: 1m 11s
800:	learn: 0.0659458	test: 0.1386050	best: 0.1385731 (793)	total: 10.8s	remaining: 1m 10s
1000:	learn: 0.0571475	test: 0.1376044	best: 0.1376044 (1000)	total: 13.7s	remaining: 1m 8s
1200:	learn: 0.0498436	test: 0.1370121	best: 0.1370121 (1200)	total: 16.5s	remaining: 1m 5s
1400:	learn: 0.0436703	test: 0.1365934	best: 0.1365910 (1397)	total: 19.3s	remaining: 1m 3s
1600:	learn: 0.0384459	test: 0.1364114	best: 0.1363946 (1562)	total: 22.3s	remaining: 1m 1s
1800:	learn: 0.0339277	test: 0.1362144	best: 0.1362065 (1782)	total: 25.1s	remaining: 58.5s
2000:	learn: 0.0296688	test: 0.1360689	best: 0.1360646 (1998)	total: 27.9s	remaining: 55.8

/Users/elenabolotin/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


200:	learn: 0.1150794	test: 0.1338441	best: 0.1338441 (200)	total: 2.65s	remaining: 1m 16s
400:	learn: 0.0873077	test: 0.1226473	best: 0.1226473 (400)	total: 5.46s	remaining: 1m 16s
600:	learn: 0.0730840	test: 0.1194963	best: 0.1194963 (600)	total: 8.22s	remaining: 1m 13s
800:	learn: 0.0627365	test: 0.1178034	best: 0.1177988 (799)	total: 11.1s	remaining: 1m 11s
1000:	learn: 0.0545357	test: 0.1168973	best: 0.1168552 (968)	total: 13.9s	remaining: 1m 9s
1200:	learn: 0.0471147	test: 0.1164398	best: 0.1164338 (1195)	total: 16.8s	remaining: 1m 6s
1400:	learn: 0.0416871	test: 0.1159057	best: 0.1158695 (1397)	total: 19.6s	remaining: 1m 4s
1600:	learn: 0.0373764	test: 0.1154661	best: 0.1154661 (1600)	total: 22.7s	remaining: 1m 2s
1800:	learn: 0.0330943	test: 0.1152415	best: 0.1152309 (1795)	total: 25.6s	remaining: 59.6s
2000:	learn: 0.0296814	test: 0.1151742	best: 0.1151542 (1975)	total: 28.5s	remaining: 56.9s
2200:	learn: 0.0265868	test: 0.1149943	best: 0.1149714 (2186)	total: 31.4s	remaining:

/Users/elenabolotin/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


200:	learn: 0.1132247	test: 0.1546199	best: 0.1546199 (200)	total: 2.59s	remaining: 1m 14s
400:	learn: 0.0856773	test: 0.1536625	best: 0.1535962 (291)	total: 5.31s	remaining: 1m 14s
600:	learn: 0.0719682	test: 0.1536813	best: 0.1532732 (453)	total: 8.14s	remaining: 1m 13s
800:	learn: 0.0611378	test: 0.1526506	best: 0.1526427 (799)	total: 11.2s	remaining: 1m 12s
1000:	learn: 0.0534322	test: 0.1523927	best: 0.1523415 (929)	total: 14s	remaining: 1m 9s
1200:	learn: 0.0475967	test: 0.1524978	best: 0.1523338 (1079)	total: 16.9s	remaining: 1m 7s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.1523338315
bestIteration = 1079

Shrink model to first 1080 iterations.
Fold 3 RMSE (log): 0.15233380182946565
0:	learn: 0.3883127	test: 0.4157134	best: 0.4157134 (0)	total: 11ms	remaining: 1m 5s


/Users/elenabolotin/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


200:	learn: 0.1143686	test: 0.1561080	best: 0.1561080 (200)	total: 2.61s	remaining: 1m 15s
400:	learn: 0.0886800	test: 0.1404858	best: 0.1404858 (400)	total: 5.34s	remaining: 1m 14s
600:	learn: 0.0760495	test: 0.1358025	best: 0.1358025 (600)	total: 8.14s	remaining: 1m 13s
800:	learn: 0.0660292	test: 0.1339171	best: 0.1339092 (794)	total: 11s	remaining: 1m 11s
1000:	learn: 0.0573152	test: 0.1327256	best: 0.1327256 (1000)	total: 13.8s	remaining: 1m 9s
1200:	learn: 0.0508402	test: 0.1319210	best: 0.1318926 (1193)	total: 16.7s	remaining: 1m 6s
1400:	learn: 0.0450966	test: 0.1311534	best: 0.1311451 (1399)	total: 19.6s	remaining: 1m 4s
1600:	learn: 0.0396553	test: 0.1305651	best: 0.1305641 (1593)	total: 22.5s	remaining: 1m 1s
1800:	learn: 0.0345776	test: 0.1301462	best: 0.1301417 (1796)	total: 25.5s	remaining: 59.4s
2000:	learn: 0.0306991	test: 0.1299965	best: 0.1299829 (1997)	total: 28.4s	remaining: 56.7s
2200:	learn: 0.0272192	test: 0.1299038	best: 0.1298859 (2196)	total: 31.3s	remaining: 

/Users/elenabolotin/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


200:	learn: 0.1159708	test: 0.1227326	best: 0.1227326 (200)	total: 2.67s	remaining: 1m 16s
400:	learn: 0.0877402	test: 0.1143666	best: 0.1143501 (399)	total: 5.42s	remaining: 1m 15s
600:	learn: 0.0741026	test: 0.1108987	best: 0.1108939 (596)	total: 8.16s	remaining: 1m 13s
800:	learn: 0.0644830	test: 0.1092386	best: 0.1092386 (800)	total: 11s	remaining: 1m 11s
1000:	learn: 0.0575705	test: 0.1082824	best: 0.1082824 (1000)	total: 13.9s	remaining: 1m 9s
1200:	learn: 0.0503839	test: 0.1079628	best: 0.1078905 (1180)	total: 16.8s	remaining: 1m 7s
1400:	learn: 0.0440774	test: 0.1074835	best: 0.1074835 (1400)	total: 19.7s	remaining: 1m 4s
1600:	learn: 0.0390866	test: 0.1070731	best: 0.1070529 (1585)	total: 22.5s	remaining: 1m 1s
1800:	learn: 0.0347232	test: 0.1068395	best: 0.1068330 (1795)	total: 25.4s	remaining: 59.3s
2000:	learn: 0.0308659	test: 0.1065244	best: 0.1065209 (1997)	total: 28.3s	remaining: 56.6s
2200:	learn: 0.0276136	test: 0.1064670	best: 0.1064522 (2183)	total: 31.2s	remaining: 

/Users/elenabolotin/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [12]:
# 6. Train final model
# =====================
final_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=200
)

final_model.fit(X, y_log, cat_features=cat_features)


0:	learn: 0.3916685	total: 6.64ms	remaining: 19.9s
200:	learn: 0.1095313	total: 1.23s	remaining: 17.1s
400:	learn: 0.0935569	total: 2.45s	remaining: 15.9s
600:	learn: 0.0840985	total: 3.68s	remaining: 14.7s
800:	learn: 0.0759938	total: 4.94s	remaining: 13.6s
1000:	learn: 0.0676504	total: 6.23s	remaining: 12.4s
1200:	learn: 0.0613307	total: 7.5s	remaining: 11.2s
1400:	learn: 0.0553866	total: 8.77s	remaining: 10s
1600:	learn: 0.0506701	total: 10s	remaining: 8.78s
1800:	learn: 0.0468492	total: 11.3s	remaining: 7.54s
2000:	learn: 0.0436328	total: 12.6s	remaining: 6.29s
2200:	learn: 0.0406446	total: 13.9s	remaining: 5.04s
2400:	learn: 0.0377817	total: 15.2s	remaining: 3.79s
2600:	learn: 0.0353364	total: 16.4s	remaining: 2.52s
2800:	learn: 0.0331850	total: 17.7s	remaining: 1.26s
2999:	learn: 0.0310609	total: 19s	remaining: 0us


In [13]:
# 7. Predict test
# =====================
test_pred_log = final_model.predict(X_test)
test_pred = np.expm1(test_pred_log)


In [14]:
# 8. Submission
# =====================
submission = sample_sub.copy()
submission["SalePrice"] = test_pred

submission.to_csv("submission_catboost_kaggle_style.csv", index=False)
print("Saved: submission_catboost_kaggle_style.csv")

Saved: submission_catboost_kaggle_style.csv


In [15]:
tableau_df = train.copy()

In [17]:
tableau_df["TotalSF"] = tableau_df["TotalBsmtSF"].fillna(0) + tableau_df["1stFlrSF"] + tableau_df["2ndFlrSF"]

In [18]:
tableau_df["HouseAge"] = tableau_df["YrSold"] - tableau_df["YearBuilt"]

In [19]:
tableau_df["Remodeled"] = (tableau_df["YearBuilt"]!= tableau_df["YearRemodAdd"]).astype(int)

In [20]:
tableau_df["QualityArea"] = tableau_df["OverallQual"] * tableau_df["GrLivArea"]

In [21]:
tableau_df["PriceSegment"] = pd.cut(
    tableau_df["SalePrice"],
    bins=[0, 150000, 300000, 1000000],
    labels=["Affordable", "Mid-Range", "Luxury"]
)

In [22]:
train_pred_log = final_model.predict(X)
train_pred = np.expm1(train_pred_log)

tableau_df["PredictedPrice"] = train_pred

In [23]:
tableau_df.to_csv(
    "house_tableau.csv",
    index=False
)